# Speak2Act × FunctionGemma — 2. Fine-tune, evaluate, convert

Fine-tunes **FunctionGemma 270M** on the data from notebook 1, compares it with the base model, and converts it to
a **`.litertlm`** file that the app runs with LiteRT-LM — then prints the Kotlin `ModelSpec` to pin it.

Recipe: Google's *Mobile Actions* notebook (full fine-tune with TRL, loss on the answer only), adapted to one
function and to the current converter (`litert-torch`, the new name of `ai-edge-torch`).

**Runtime — free:** *Runtime ▸ Change runtime type ▸ T4 GPU* (Colab's free tier). The T4 has no bfloat16, so
FunctionGemma trains in float32 — the same result, just slower: about 1–1.5 hours for the whole notebook. The
fine-tuned model is saved to Drive right after training, so if the free session ends, a new one continues from there.

**Before you start:**
1. Accept the FunctionGemma license on https://huggingface.co/google/functiongemma-270m-it (instant).
2. Add your Hugging Face token as a Colab secret named **`HF_TOKEN`** (🔑 in the left sidebar).
3. Notebook 1's files are in `MyDrive/speak2act/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# The versions Google's FunctionGemma fine-tuning notebook was verified with.
%pip install -q -U transformers==4.57.1 trl==0.25.1 datasets==4.4.1

In [ ]:
#@title Settings
import torch
DATA_DIR = "/content/drive/MyDrive/speak2act"   #@param {type:"string"}
BASE_MODEL = "google/functiongemma-270m-it"
TUNED_DIR = f"{DATA_DIR}/speak2act-functiongemma-hf"   # the fine-tuned checkpoint, on Drive
CHECKPOINT_DIR = "/content/checkpoints"          # the trainer's scratch space
BASE_DIR = "/content/functiongemma-base"
OUT_NAME = "speak2act-functiongemma-v1"         #@param {type:"string"}
EPOCHS = 2                                       #@param {type:"integer"}
LEARNING_RATE = 2e-5                             #@param {type:"number"}
BATCH_SIZE = 4                                   #@param {type:"integer"}
GRAD_ACCUMULATION = 8                            #@param {type:"integer"}
CONVERT_BASE_TOO = True                          #@param {type:"boolean"}
EVAL_LIMIT = 200                                 #@param {type:"integer"}
EVAL_NEG_LIMIT = 60                              #@param {type:"integer"}
MAX_NEW_TOKENS = 96          # the app's maxOutputToken (plan §8.5.3): a full call is ~50 tokens
# bfloat16 only in hardware (compute capability 8+): torch.cuda.is_bf16_supported() says True on a T4 too.
BF16 = torch.cuda.get_device_capability()[0] >= 8
DTYPE = torch.bfloat16 if BF16 else torch.float32   # the free T4: float32 (Gemma models overflow plain float16)
print(torch.cuda.get_device_name(0), "→", DTYPE)

In [ ]:
from google.colab import drive, userdata
from huggingface_hub import login
drive.mount("/content/drive")
login(userdata.get("HF_TOKEN"))

In [ ]:
TOOL = {
  "description": "Prepares a money transaction for the user to confirm.",
  "name": "executeTransaction",
  "parameters": {
    "properties": {
      "action": {
        "description": "Type of the transaction.",
        "enum": [
          "pay",
          "request",
          "split"
        ],
        "type": "string"
      },
      "amount": {
        "description": "Amount of the transaction.",
        "type": "number"
      },
      "currency": {
        "description": "Currency of the transaction, e.g. CHF or EUR.",
        "type": "string"
      },
      "description": {
        "description": "The reason for the transaction.",
        "type": "string"
      },
      "person": {
        "description": "Contact to whom the transaction is addressed.",
        "type": "string"
      }
    },
    "required": [
      "action",
      "amount",
      "currency",
      "person"
    ],
    "type": "object"
  }
}
SYSTEM_PROMPT = "You are a model that can do function calling with the following functions"

In [ ]:
%%writefile s2a_eval.py
"""Speak2Act × FunctionGemma — evaluation helpers (pure Python). Used by notebook 02 and tested locally.

- litertlm_declaration(): renders a tool exactly like LiteRT-LM v0.18.1 does on the phone (fc_tool_format_utils.cc):
  JSON key order kept, JSON-schema type names upper-cased, strings wrapped in the escape token.
- litertlm_prompt(): the whole prompt as LiteRT-LM 0.18.1's built-in FunctionGemma template renders it on the
  phone. It differs from the Hugging Face chat template in two places: a blank line between the system instruction
  and the tool declaration, and the Hugging Face template silently skips parameters named like JSON-schema keywords
  (`description`, `type`, …) — LiteRT-LM declares them all. Notebook 02 trains and evaluates on the phone's version;
  hf_template_view() / hf_matches_phone() check that the Hugging Face template agrees on everything else.
- parse_calls(): a strict reader for FunctionGemma's function calls (numbers as Decimal; malformed calls are skipped).
- score() / score_calls(): "core" (action, amount, currency, person: what the app's guard and mapper need) and
  "exact" (core + the reason). Like the app, only a single call to executeTransaction counts.
"""
import copy
import json
import re
from decimal import Decimal, InvalidOperation

ESC = "<escape>"
CALL_START, CALL_END = "<start_function_call>", "<end_function_call>"
DECL_START, DECL_END = "<start_function_declaration>", "<end_function_declaration>"
SYSTEM_PROMPT = "You are a model that can do function calling with the following functions"
FUNCTION_NAME = "executeTransaction"
DEFAULT_CURRENCY = "CHF"
TYPE_NAMES = {"string", "number", "integer", "object", "array", "boolean", "null"}
# LiteRT-LM 0.18.1's FunctionGemma template: "<start_of_turn>developer\n" + system instruction + "\n\n" + declarations.
# The Hugging Face template has no blank line there.
PHONE_SEPARATOR = "\n\n"
# The Hugging Face chat template's format_parameters skips properties with these names (its "standard_keys");
# LiteRT-LM's FormatToolAsFc declares every property.
HF_SKIPPED_PROPERTY_NAMES = {"description", "type", "properties", "required", "nullable"}


# ---------------------------------------------------------------- what the phone renders
def _upper_types(value):
    if isinstance(value, dict):
        out = {}
        for k, v in value.items():
            if k == "type" and isinstance(v, str) and v in TYPE_NAMES:
                out[k] = v.upper()
            else:
                out[k] = _upper_types(v)
        return out
    if isinstance(value, list):
        return [_upper_types(v) for v in value]
    return value


def _fc(value):
    if value is None:
        return "null"
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, (int, float)):
        return json.dumps(value)
    if isinstance(value, str):
        return f"{ESC}{value}{ESC}"
    if isinstance(value, list):
        return "[" + ",".join(_fc(v) for v in value) + "]"
    if isinstance(value, dict):
        return "{" + ",".join(f"{k}:{_fc(v)}" for k, v in value.items()) + "}"
    raise TypeError(type(value))


def litertlm_declaration(tool):
    """`declaration:NAME{…}` as LiteRT-LM renders an OpenApiTool's JSON (without the declaration tokens)."""
    fields = _upper_types(tool)
    name = fields.pop("name")
    return f"declaration:{name}{_fc(fields)}"


def litertlm_prompt(system_prompt, tool, sentence):
    """The prompt LiteRT-LM 0.18.1 renders on the phone for one command (after the BOS token): the developer turn
    (system instruction, blank line, declaration), the user's sentence, and the start of the model's turn."""
    return ("<start_of_turn>developer\n" + system_prompt.strip() + PHONE_SEPARATOR
            + DECL_START + litertlm_declaration(tool) + DECL_END + "<end_of_turn>\n"
            + "<start_of_turn>user\n" + sentence.strip() + "<end_of_turn>\n"
            + "<start_of_turn>model\n")


def to_phone_prompt(hf_prompt, system_prompt):
    """A prompt rendered by the Hugging Face chat template, changed into what the phone renders: the blank line
    between the system instruction and the tool declaration. Raises if the prompt doesn't have that shape."""
    head = system_prompt.strip() + DECL_START
    if hf_prompt.count(head) != 1:
        raise ValueError("expected the system instruction right before the tool declaration, exactly once")
    return hf_prompt.replace(head, system_prompt.strip() + PHONE_SEPARATOR + DECL_START)


def hf_template_view(tool):
    """The tool as the Hugging Face chat template declares it: without parameters named like JSON-schema keywords
    (our `description` parameter!). Nested object parameters are filtered the same way."""
    view = copy.deepcopy(tool)

    def strip(schema):
        props = schema.get("properties")
        if not isinstance(props, dict):
            return
        for key in list(props):
            if key in HF_SKIPPED_PROPERTY_NAMES:
                del props[key]
            elif isinstance(props[key], dict):
                strip(props[key])

    if isinstance(view.get("parameters"), dict):
        strip(view["parameters"])
    return view


def hf_matches_phone(hf_prompt, system_prompt, tool, sentence, bos=""):
    """True when a prompt rendered by the Hugging Face template equals the phone's (litertlm_prompt), except for the
    two known differences: the blank line after the system prompt and the parameters the template skips."""
    phone = litertlm_prompt(system_prompt, tool, sentence)
    hf_expected = phone.replace(litertlm_declaration(tool), litertlm_declaration(hf_template_view(tool)))
    return to_phone_prompt(hf_prompt, system_prompt) == bos + hf_expected


def declarations_in(prompt):
    """The declaration blocks inside a rendered prompt."""
    return re.findall(re.escape(DECL_START) + r"(.*?)" + re.escape(DECL_END), prompt, re.S)


def is_alphabetical(value):
    """True when every JSON object has its keys in alphabetical order (what the training template produces)."""
    if isinstance(value, dict):
        return list(value) == sorted(value) and all(is_alphabetical(v) for v in value.values())
    if isinstance(value, list):
        return all(is_alphabetical(v) for v in value)
    return True


# ---------------------------------------------------------------- what the model says
_NUMBER = re.compile(r"-?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?")
_LITERAL = re.compile(r"true|false|null")
_KEY = re.compile(r"[A-Za-z_]\w*")


def _parse_value(text, i):
    if text.startswith(ESC, i):
        end = text.find(ESC, i + len(ESC))
        if end < 0:
            raise ValueError("unterminated string")
        return text[i + len(ESC):end], end + len(ESC)
    if text.startswith("{", i):
        return _parse_object(text, i)
    if text.startswith("[", i):
        return _parse_array(text, i)
    m = _NUMBER.match(text, i)
    if m:
        return Decimal(m.group(0)), m.end()
    m = _LITERAL.match(text, i)
    if m:
        return {"true": True, "false": False, "null": None}[m.group(0)], m.end()
    raise ValueError(f"bad value at {i}: {text[i:i + 20]!r}")


def _parse_array(text, i):
    items, i = [], i + 1
    while True:
        if i >= len(text):
            raise ValueError("unterminated array")
        if text[i] == "]":
            return items, i + 1
        value, i = _parse_value(text, i)
        items.append(value)
        if i < len(text) and text[i] == ",":
            i += 1
        elif i >= len(text) or text[i] != "]":
            raise ValueError(f"expected ',' or ']' at {i}")


def _parse_object(text, i):
    out, i = {}, i + 1
    while True:
        if i >= len(text):
            raise ValueError("unterminated object")
        if text[i] == "}":
            return out, i + 1
        colon = text.find(":", i)
        if colon < 0:
            raise ValueError("missing ':'")
        key = text[i:colon].strip()
        if not _KEY.fullmatch(key):
            raise ValueError(f"bad key {key!r}")
        value, i = _parse_value(text, colon + 1)
        out[key] = value
        if i < len(text) and text[i] == ",":
            i += 1
        elif i >= len(text) or text[i] != "}":
            raise ValueError(f"expected ',' or '}}' at {i}")


def parse_calls(output):
    """[{"name": …, "arguments": {…}}] for every well-formed call in the model output. A call must be closed
    (`<end_function_call>`), and its argument object must be complete and the whole body; otherwise it's skipped."""
    calls = []
    for raw in re.findall(re.escape(CALL_START) + r"(.*?)" + re.escape(CALL_END), output, re.S):
        m = re.fullmatch(r"\s*call:([A-Za-z_][\w.-]*)(\{.*\})\s*", raw, re.S)
        if not m:
            continue
        try:
            args, end = _parse_object(m.group(2), 0)
        except ValueError:
            continue
        if end == len(m.group(2)):
            calls.append({"name": m.group(1), "arguments": args})
    return calls


# ---------------------------------------------------------------- scoring
REASON_STOPWORDS = {"the", "a", "an", "for", "of", "my", "our", "to", "his", "her", "their"}


def _decimal(value):
    try:
        return Decimal(str(value))
    except (InvalidOperation, ValueError, TypeError):
        return None


def _words(text):
    return re.findall(r"[^\W_]+", str(text or "").lower())


def _core(args):
    """What the app's mapper turns into a Transaction (a missing currency becomes CHF, like the app)."""
    currency = str(args.get("currency") or DEFAULT_CURRENCY).strip().upper()
    return (str(args.get("action", "")).strip().lower(), _decimal(args.get("amount")), currency,
            " ".join(_words(args.get("person"))))


def _reason(args):
    return [w for w in _words(args.get("description")) if w not in REASON_STOPWORDS]


def score_calls(target_args, calls):
    """target_args: the expected arguments, or None for "no call". calls: parsed calls, [{"name", "arguments"}].
    Returns {"core": bool, "exact": bool, "kind": str}. Like the app, only exactly one executeTransaction call counts."""
    if target_args is None:
        ok = not calls
        return {"core": ok, "exact": ok, "kind": "no_call_ok" if ok else "unexpected_call"}
    if not calls:
        return {"core": False, "exact": False, "kind": "missing_call"}
    if len(calls) > 1:
        return {"core": False, "exact": False, "kind": "several_calls"}
    if calls[0]["name"] != FUNCTION_NAME:
        return {"core": False, "exact": False, "kind": "wrong_function"}
    got = calls[0]["arguments"]
    core = _core(got) == _core(target_args)
    exact = core and _reason(got) == _reason(target_args)
    return {"core": core, "exact": exact, "kind": "ok" if core else "wrong_args"}


def score(target_args, output):
    """score_calls() on the calls in a raw model output."""
    return score_calls(target_args, parse_calls(output))


# ---------------------------------------------------------------- golden set (plan §10.1, G1–G10): typed, never trained on
# What the model alone should answer. On the phone, G1 and G2 never reach the model (the grammar answers), and the
# guard refuses G10 whatever the model says (two amounts).
GOLDEN_EXPECTED = [
    ("Request 10 euros from Maria for the taxi bill.",
     {"action": "request", "amount": 10, "currency": "EUR", "person": "Maria", "description": "the taxi bill"}),
    ("Send twenty five francs to Stefan.", {"action": "pay", "amount": 25, "currency": "CHF", "person": "Stefan"}),
    ("I owe Stefan ten francs for the pizza.",
     {"action": "pay", "amount": 10, "currency": "CHF", "person": "Stefan", "description": "pizza"}),
    ("Split 60 euros with Maria for dinner.",
     {"action": "split", "amount": 60, "currency": "EUR", "person": "Maria", "description": "dinner"}),
    ("Could you get 25 bucks from Tamara for lunch?",
     {"action": "request", "amount": 25, "currency": "USD", "person": "Tamara", "description": "lunch"}),
    ("I paid for the taxi, Maria owes me ten euros.",
     {"action": "request", "amount": 10, "currency": "EUR", "person": "Maria", "description": "taxi"}),
    ("Schick Stefan 10 Franken für die Pizza.",
     {"action": "pay", "amount": 10, "currency": "CHF", "person": "Stefan", "description": "Pizza"}),
    ("Remind me to call Stefan tomorrow.", None),
    ("Send some money to Stefan.", None),
    ("Transfer 100 francs to Stefan and 50 to Maria.", None),
]

In [ ]:
import importlib, s2a_eval as e
importlib.reload(e)

## Parity check: does the phone show the model what training shows it?
On the phone, LiteRT-LM 0.18.1 renders the prompt itself: the tool declaration from the app's JSON (key order kept,
**every** parameter), inside its built-in FunctionGemma template — which puts a **blank line** between the system
prompt and the declaration. The Hugging Face template differs in exactly three ways: it sorts the keys, it has no
blank line, and it silently **skips parameters named like JSON-schema keywords** — our `description` parameter
(its `standard_keys`). So this notebook builds every training and evaluation prompt the way the phone does
(`e.litertlm_prompt`), and takes only the answers from the Hugging Face template. This cell checks that the
Hugging Face template agrees on everything else. If it fails, fix the key order of the JSON — in **both** the app
and this notebook — before you train. After the conversion, a later cell repeats the check with LiteRT-LM itself.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
TOOLS = [{"type": "function", "function": TOOL}]
PROBE = "I owe Stefan ten francs"
hf_probe = tokenizer.apply_chat_template(
    [{"role": "developer", "content": SYSTEM_PROMPT}, {"role": "user", "content": PROBE}],
    tools=TOOLS, tokenize=False, add_generation_prompt=True)
phone_probe = e.litertlm_prompt(SYSTEM_PROMPT, TOOL, PROBE)
print(phone_probe)
assert e.is_alphabetical(TOOL), "the tool JSON must have alphabetical keys"
skipped = sorted(set(TOOL["parameters"]["properties"]) & e.HF_SKIPPED_PROPERTY_NAMES)
print("Declared on the phone, skipped by the Hugging Face template:", skipped)
hf_view = e.litertlm_declaration(e.hf_template_view(TOOL))
assert e.declarations_in(hf_probe) == [hf_view], (e.declarations_in(hf_probe), hf_view)
assert e.hf_matches_phone(hf_probe, SYSTEM_PROMPT, TOOL, PROBE, bos=tokenizer.bos_token), hf_probe
print("✅ Parity: the Hugging Face template agrees with the phone on everything but the known differences —")
print("   training and evaluation use the phone's prompt, with every parameter declared.")

## Load the data (prompt / completion)

In [ ]:
import json
from datasets import load_dataset

files = {"train": f"{DATA_DIR}/speak2act_fc_train.jsonl", "eval": f"{DATA_DIR}/speak2act_fc_eval.jsonl"}
raw = load_dataset("text", data_files=files, encoding="utf-8")   # as text: keeps each row's JSON exactly as written

def apply_format(sample):
    row = json.loads(sample["text"])
    assert row["tools"] == TOOLS, "the dataset was built with a different tool JSON"
    sentence = row["messages"][1]["content"]
    full = tokenizer.apply_chat_template(row["messages"], tools=row["tools"], tokenize=False, add_generation_prompt=False)
    hf_prompt = tokenizer.apply_chat_template(row["messages"][:-1], tools=row["tools"], tokenize=False, add_generation_prompt=True)
    assert full.startswith(hf_prompt)
    assert e.hf_matches_phone(hf_prompt, SYSTEM_PROMPT, TOOL, sentence, bos=tokenizer.bos_token), hf_prompt
    # The prompt exactly as the phone renders it — every parameter declared — and without the "<bos>" text: the
    # tokenizer adds the one BOS token. The answer (the call, or "no call") comes from the chat template.
    return {"prompt": e.litertlm_prompt(SYSTEM_PROMPT, TOOL, sentence), "completion": full[len(hf_prompt):]}

data = raw.map(apply_format, remove_columns=["text"])
lengths = [len(tokenizer.tokenize(x["prompt"] + x["completion"])) for x in data["train"]]
max_token_count = max(lengths) + 64
print(data, "\nlongest example:", max(lengths), "tokens → max_length", max_token_count)
print(data["train"][0]["prompt"] + "⟶" + data["train"][0]["completion"])    # what slide "What FunctionGemma actually sees" shows

## Evaluation
Greedy decoding, like the app (`topK = 1`), on the phone's prompt and with the app's output limit. Three groups:
**eval · call** (should call the function), **eval · no call** (must not), and the **golden set** G1–G10 (the talk's
sentences, never trained on). **core** = exactly one call, with the right action, amount, currency and person (what
the app's guard and mapper need); **exact** also checks the reason. The eval split never shares a seed with training.

In [ ]:
import pandas as pd
from transformers import AutoModelForCausalLM

def eval_cases():
    calls, no_calls = [], []
    for line in raw["eval"]["text"]:
        row = json.loads(line)
        answer = row["messages"][2]
        if "tool_calls" in answer:
            calls.append(("eval · call", row["messages"][1]["content"], answer["tool_calls"][0]["function"]["arguments"]))
        else:
            no_calls.append(("eval · no call", row["messages"][1]["content"], None))
    assert calls and no_calls, "the eval split needs both kinds of examples"
    return calls[:EVAL_LIMIT] + no_calls[:EVAL_NEG_LIMIT] + [("golden", s, t) for s, t in e.GOLDEN_EXPECTED]

CASES = eval_cases()
tokenizer.padding_side = "left"      # batched generation

def evaluate(model, label, batch_size=16):
    model.eval()
    records = []
    for start in range(0, len(CASES), batch_size):
        chunk = CASES[start:start + batch_size]
        # Exactly what the app sends (e.litertlm_prompt); the tokenizer adds the BOS token, like in training.
        prompts = [e.litertlm_prompt(SYSTEM_PROMPT, TOOL, sentence) for _, sentence, _ in chunk]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
        for (split, sentence, target), ids in zip(chunk, out[:, inputs["input_ids"].shape[-1]:]):
            output = tokenizer.decode(ids, skip_special_tokens=False)
            records.append({"model": label, "split": split, "sentence": sentence, "output": output,
                            **e.score(target, output)})
    df = pd.DataFrame(records)
    print(df.groupby("split")[["core", "exact"]].mean().round(3))
    return df

### The base model — the "before"

In [ ]:
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=DTYPE, device_map="auto", attn_implementation="eager")
base_scores = evaluate(base, "base")
del base
torch.cuda.empty_cache()

## Fine-tune
The fine-tuned model goes to `TUNED_DIR` on Drive. **If the free session ends after this cell**, open a new one and
run the cells from Settings down to the evaluation cell, plus the base model's (2 minutes) — then continue below
this cell. Nothing needs to train twice.

In [ ]:
from trl import SFTConfig, SFTTrainer

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=DTYPE, device_map="auto", attn_implementation="eager")
model.config.pad_token_id = tokenizer.pad_token_id

args = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    max_length=max_token_count,
    packing=False,
    completion_only_loss=True,          # learn the call, not the prompt
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    bf16=BF16,                          # False on the free T4: float32
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="no",                 # the final model goes to Drive below
    report_to="none",
)
trainer = SFTTrainer(model=model, args=args, train_dataset=data["train"], eval_dataset=data["eval"],
                     processing_class=tokenizer)
result = trainer.train()
trainer.model.config.use_cache = True    # gradient checkpointing switched it off; generation needs it
trainer.save_model(TUNED_DIR)            # on Drive: a new session can continue from here
tokenizer.save_pretrained(TUNED_DIR)
print(f"Training took {result.metrics['train_runtime'] / 60:.1f} min on {torch.cuda.get_device_name(0)}; "
      f"saved to {TUNED_DIR}")
del model, trainer
torch.cuda.empty_cache()

## Evaluate the fine-tuned model — before vs after

In [ ]:
tuned = AutoModelForCausalLM.from_pretrained(TUNED_DIR, dtype=DTYPE, device_map="auto", attn_implementation="eager")
tuned_scores = evaluate(tuned, "fine-tuned")
frames = [base_scores, tuned_scores] if "base_scores" in globals() else [tuned_scores]
summary = pd.concat(frames).groupby(["split", "model"])[["core", "exact"]].mean().round(3)
print(summary)
summary.to_csv(f"{DATA_DIR}/eval_summary.csv")

In [ ]:
# What does the fine-tuned model still get wrong?
pd.set_option("display.max_colwidth", 120)
tuned_scores[~tuned_scores["core"]][["split", "sentence", "kind", "output"]].head(30)

## Convert to `.litertlm`
The same conversion as Google's notebook, with the renamed package (`litert-torch`). The metadata marks the file as
**FunctionGemma**, so LiteRT-LM parses the calls into `Message.toolCalls` by itself. Context: 1,024 tokens of KV cache
(the prompt with the declaration is a few hundred). If Colab asks to restart after the install: restart, run the cells
from **Settings** down to `import s2a_eval` again (they're quick), then continue here — the fine-tuned model is on
Drive. The conversion runs on the CPU; the free runtime's 12 GB of RAM are enough for a 270M model.

In [ ]:
!pip uninstall -y -q tensorflow
%pip install -q litert-torch==0.10.0

In [ ]:
import os, shutil, hashlib
from huggingface_hub import hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer
from litert_torch.generative.examples.gemma3 import gemma3
from litert_torch.generative.utilities import converter
from litert_torch.generative.utilities.export_config import ExportConfig
from litert_torch.generative.layers import kv_cache

LLM_METADATA = r"""start_token: {
    token_ids: {
        ids: [ 2 ]
    }
}
stop_tokens: {
    token_str: "<end_of_turn>"
}
stop_tokens: {
    token_str: "<start_function_response>"
}
llm_model_type: {
    function_gemma: {}
}
"""
OUT_DIR = "/content/litertlm"
os.makedirs(OUT_DIR, exist_ok=True)
metadata_path = os.path.join(OUT_DIR, "base_llm_metadata.textproto")
open(metadata_path, "w").write(LLM_METADATA)

def convert(checkpoint_dir, name):
    for old in os.listdir(OUT_DIR):            # rerunnable: remove this model's files from an earlier run
        if old.startswith(name) and old.endswith(".litertlm"):
            os.remove(os.path.join(OUT_DIR, old))
    tokenizer_model = os.path.join(checkpoint_dir, "tokenizer.model")
    if not os.path.exists(tokenizer_model):
        hf_hub_download(BASE_MODEL, "tokenizer.model", local_dir=checkpoint_dir)
    export_config = ExportConfig()
    export_config.kvcache_layout = kv_cache.KV_LAYOUT_TRANSPOSED
    export_config.mask_as_input = True
    converter.convert_to_litert(
        gemma3.build_model_270m(checkpoint_dir),
        output_path=OUT_DIR,
        output_name_prefix=name,
        prefill_seq_len=256,
        kv_cache_max_len=1024,
        quantize="dynamic_int8",
        export_config=export_config,
        tokenizer_model_path=tokenizer_model,
        base_llm_metadata_path=metadata_path,
        output_format="litertlm",
    )
    produced = [f for f in os.listdir(OUT_DIR) if f.startswith(name) and f.endswith(".litertlm")]
    assert len(produced) == 1, produced
    final = os.path.join(OUT_DIR, f"{name}.litertlm")
    os.replace(os.path.join(OUT_DIR, produced[0]), final)
    return final

tuned_file = convert(TUNED_DIR, OUT_NAME)
base_file = None
if CONVERT_BASE_TOO:   # the "before" of the before/after screenshots (plan §8.10)
    AutoModelForCausalLM.from_pretrained(BASE_MODEL).save_pretrained(BASE_DIR)
    AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(BASE_DIR)
    base_file = convert(BASE_DIR, "functiongemma-270m-base")
print(tuned_file, base_file)

## Check on the app's runtime: LiteRT-LM 0.18.1, in Colab
`litert-lm-api` is LiteRT-LM's Python API — the same C++ runtime, chat template and call parser as
`litertlm-android` 0.18.1 in the app. This cell loads the converted file on the CPU, sets up the conversation like
`FunctionGemmaTransactionModel` does (system prompt, the same tool JSON, greedy, no automatic tool calling, the same
output limit), checks that a fresh conversation renders **exactly** the training prompt, and runs the golden set.
It's the last check before the phone: the template, the conversion metadata and the call parsing, end to end. If the install fails, skip it — the phone check (plan §8.5.5) covers the same.

In [ ]:
%pip install -q litert-lm-api==0.18.1

In [ ]:
import litert_lm
import pandas as pd

class TransactionTool(litert_lm.Tool):
    """The app's tool: the same JSON, wrapped like LiteRT-LM's Kotlin API wraps an OpenApiTool."""
    def get_tool_description(self):
        return {"type": "function", "function": TOOL}
    def execute(self, param):
        raise RuntimeError("proposed, never executed")

def text(t):
    return [{"type": "text", "text": t}]

def new_conversation(engine):
    """Set up like FunctionGemmaTransactionModel: system prompt, the tool, greedy, propose only, the same limit."""
    return engine.create_conversation(
        messages=[{"role": "system", "content": text(SYSTEM_PROMPT)}],   # = ConversationConfig.systemInstruction
        tools=[TransactionTool()],
        automatic_tool_calling=False,
        sampler_config=litert_lm.SamplerConfig(top_k=1, top_p=1.0, temperature=1.0),
        max_output_tokens=MAX_NEW_TOKENS,
    )

def runtime_golden(model_file, label, probe=None):
    """The golden set on LiteRT-LM; with a probe sentence, also the prompt a fresh conversation renders for it."""
    rows, rendered = [], None
    engine = litert_lm.Engine(model_file, backend=litert_lm.Backend.CPU())
    try:
        if probe:
            conversation = new_conversation(engine)   # a fresh conversation renders its whole first prompt
            rendered = conversation.render_message_to_string({"role": "user", "content": text(probe)})
            conversation.close()
        for sentence, target in e.GOLDEN_EXPECTED:
            conversation = new_conversation(engine)
            reply = conversation.send_message({"role": "user", "content": text(sentence)})
            calls = [{"name": c.name, "arguments": dict(c.arguments)} for c in reply.tool_calls]
            rows.append({"model": label, "sentence": sentence, "calls": calls, **e.score_calls(target, calls)})
            conversation.close()
    finally:
        engine.close()
    return rows, rendered

PROBE = "I owe Stefan ten francs"
expected = e.litertlm_prompt(SYSTEM_PROMPT, TOOL, PROBE)
tuned_rows, rendered = runtime_golden(tuned_file, "fine-tuned", probe=PROBE)
base_rows = runtime_golden(base_file, "base")[0] if base_file else []
# LiteRT-LM renders bos_token as "" and adds the start token itself; strip "<bos>" in case a version renders it.
prompt_ok = bool(rendered) and rendered.removeprefix("<bos>") == expected
print("LiteRT-LM renders:\n" + (rendered or "<nothing>"))
print("✅ the whole prompt matches training" if prompt_ok else "⚠️ differs from the training prompt, which is:\n" + expected)
runtime = pd.DataFrame(tuned_rows + base_rows)
print(runtime.groupby("model")[["core", "exact"]].sum())
display(runtime[["model", "sentence", "kind", "calls"]])
assert prompt_ok, "the phone would see a different prompt than training — don't publish (guide §8)"

## Pin it: size, SHA-256 and the Kotlin snippet

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def pin(path):
    shutil.copy(path, DATA_DIR)
    size, digest = os.path.getsize(path), sha256(path)
    print(f"{os.path.basename(path)}: {size:,} bytes ({size / 1e6:.0f} MB) — copied to {DATA_DIR}")
    return size, digest

size, digest = pin(tuned_file)
print(f"""
// FunctionGemmaModel.TUNED — the fine-tuned model (plan §9.4)
val TUNED = ModelSpec(
    fileName = "{os.path.basename(tuned_file)}",
    url = "<public URL from plan §9.3>",
    sizeBytes = {size}L,
    sha256 = "{digest}",
    isStale = {{ it.startsWith("speak2act-functiongemma-") && it.endsWith(".litertlm") }},
)""")
if base_file:
    size, digest = pin(base_file)
    print(f"""
// FunctionGemmaModel.BASE — the untuned model, ONLY for the before/after comparison (plan §8.10). Never as TUNED.
val BASE = ModelSpec("{os.path.basename(base_file)}", url = "", sizeBytes = {size}L, sha256 = "{digest}")""")

## Publish (optional, after the human gate H5)
The app downloads the model anonymously, so it needs a **public** URL. This cell creates a public Hugging Face model
repo with the `.litertlm`, a model card and the **NOTICE** the Gemma Terms of Use ask for. Read the terms first —
this isn't legal advice. Needs a token with *write* access.

In [ ]:
PUBLISH = False        #@param {type:"boolean"}
HF_REPO = ""           #@param {type:"string"}
if PUBLISH:
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(HF_REPO, repo_type="model", private=False, exist_ok=True)
    notice = "Gemma is provided under and subject to the Gemma Terms of Use found at ai.google.dev/gemma/terms\n"
    card = f"""---
license: gemma
base_model: {BASE_MODEL}
tags: [function-calling, litert-lm, on-device, speak2act]
---
# {OUT_NAME}

FunctionGemma 270M **fine-tuned** (modified) for one function, `executeTransaction`, used by the open-source demo app
[Speak2Act](https://github.com/stevan-milovanovic/Speak2Act). LiteRT-LM file, `dynamic_int8`, KV cache 1,024.

- Training data: synthetic — seeds labelled by code, sentences written by Gemma 4 E2B.
- Evaluation (eval split / golden set, core accuracy): see `eval_summary.csv`.
- **Demo only.** The app never moves money without the user's explicit confirmation.

Gemma is provided under and subject to the Gemma Terms of Use found at ai.google.dev/gemma/terms. Use is subject to
the Gemma Prohibited Use Policy (ai.google.dev/gemma/prohibited_use_policy).
"""
    open("README.md", "w").write(card)
    open("NOTICE", "w").write(notice)
    for path, name in [(tuned_file, os.path.basename(tuned_file)), ("README.md", "README.md"), ("NOTICE", "NOTICE"),
                       (f"{DATA_DIR}/eval_summary.csv", "eval_summary.csv")]:
        api.upload_file(path_or_fileobj=path, path_in_repo=name, repo_id=HF_REPO)
    print("URL for the app:", f"https://huggingface.co/{HF_REPO}/resolve/main/{os.path.basename(tuned_file)}")

## Next
1. Try it on the phone before publishing: `adb push` (plan §8.5.5). The first command logs `llm_preface=…`: it must
   match the parity cell's output above.
2. Publish (above, or a GitHub Release asset) and pin the URL, size and SHA-256 in `FunctionGemmaModel.TUNED`
   (plan §9.4).
3. Note for the deck: GPU, training minutes, examples, and the before/after numbers above.